## Association Mining with FP-Growth

This notebook demonstrates how to use the FP-Growth algorithm to find frequent itemsets and association rules in the flight delay dataset.

In [141]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
import warnings
warnings.filterwarnings("ignore")

In [142]:
df = pd.read_csv('../data/cleaned_flight_data.csv')
df.head()

,scheduled_time,airline,flight_number,destination_or_origin,status,counter,actual_time,register,aircraft,airport,...,scheduled_day_of_week,is_weekend,is_Nourooz_4,is_Nourooz_13,Normal_holiday,scheduled_season,scheduled_time_of_day,Early_OnTime_Late_Indicator,Actual_Day_Matches_Scheduled_Day,Scheduled_Hour_of_Day
0,2025-05-28 06:50:00,کاسپین,CPN024,مشهد,پرواز كرد,"12, 13",2025-05-28 07:21:00,EPCPU,MD83,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,6
1,2025-05-28 07:00:00,اطلس ایر,ATS8271,كيش,پرواز كرد,"16, 17",2025-05-28 07:39:00,EPSAP,MD83,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
2,2025-05-28 07:00:00,آتا,TBZ5700,سیرجان,پرواز كرد,"26, 27",2025-05-28 07:27:00,EPTAH,737-700,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
3,2025-05-28 07:05:00,پويا,PYA2350,اصفهان,پرواز كرد,14,2025-05-28 07:23:00,EPPUN,EMB145,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
4,2025-05-28 07:05:00,ایران ایر,IRA311,اهواز,پرواز كرد,"5, 6",2025-05-28 07:29:00,EPIEQ,A319,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7


In [143]:
air = df[df['airport'] == 'فرودگاه مهرآباد'].copy()
print(f"percentage: {(air.shape[0])/df.shape[0]}")

percentage: 0.9148292261132728


In [144]:
df.columns

Index(['scheduled_time', 'airline', 'flight_number', 'destination_or_origin',
       'status', 'counter', 'actual_time', 'register', 'aircraft', 'airport',
       'flight_type', 'scheduled', 'actual', 'scheduled_datetime',
       'actual_datetime', 'delay', 'delay_minutes', 'scheduled_day_of_week',
       'is_weekend', 'is_Nourooz_4', 'is_Nourooz_13', 'Normal_holiday',
       'scheduled_season', 'scheduled_time_of_day',
       'Early_OnTime_Late_Indicator', 'Actual_Day_Matches_Scheduled_Day',
       'Scheduled_Hour_of_Day'],
      dtype='object')

In [145]:
# Select features for association mining
# df_assoc = df[['airline', 'destination_or_origin', 'aircraft', 'airport', 'scheduled_day_of_week', 'scheduled_season', 'scheduled_time_of_day', 'Early_OnTime_Late_Indicator']].copy()
df_assoc = air[['airline', 'destination_or_origin', 'aircraft', 'scheduled_day_of_week', 'scheduled_season', 'scheduled_time_of_day', 'Early_OnTime_Late_Indicator']].copy()
# Drop rows with missing values
df_assoc.dropna(inplace=True)

df.head()

,scheduled_time,airline,flight_number,destination_or_origin,status,counter,actual_time,register,aircraft,airport,...,scheduled_day_of_week,is_weekend,is_Nourooz_4,is_Nourooz_13,Normal_holiday,scheduled_season,scheduled_time_of_day,Early_OnTime_Late_Indicator,Actual_Day_Matches_Scheduled_Day,Scheduled_Hour_of_Day
0,2025-05-28 06:50:00,کاسپین,CPN024,مشهد,پرواز كرد,"12, 13",2025-05-28 07:21:00,EPCPU,MD83,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,6
1,2025-05-28 07:00:00,اطلس ایر,ATS8271,كيش,پرواز كرد,"16, 17",2025-05-28 07:39:00,EPSAP,MD83,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
2,2025-05-28 07:00:00,آتا,TBZ5700,سیرجان,پرواز كرد,"26, 27",2025-05-28 07:27:00,EPTAH,737-700,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
3,2025-05-28 07:05:00,پويا,PYA2350,اصفهان,پرواز كرد,14,2025-05-28 07:23:00,EPPUN,EMB145,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7
4,2025-05-28 07:05:00,ایران ایر,IRA311,اهواز,پرواز كرد,"5, 6",2025-05-28 07:29:00,EPIEQ,A319,فرودگاه مهرآباد,...,Wednesday,0,0,0,0,Spring,Morning,Late,1,7


In [146]:
# Convert the dataframe into a list of transactions
transactions = df_assoc.to_numpy().tolist()

In [147]:
# Use TransactionEncoder to transform the data into a one-hot encoded format
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

In [148]:
# Run the FP-Growth algorithm
frequent_itemsets = fpgrowth(df_onehot, min_support=0.01, use_colnames=True)

In [149]:
# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# Display the rules
rules.sort_values(by='lift', ascending=False).head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
7762,(تابان),"(Spring, MD.88)",0.011815,0.015595,0.011815,1.000000,64.121212,1.0,0.01163,inf,0.996174,0.757576,1.000000,0.878788
7759,"(Spring, MD.88)",(تابان),0.015595,0.011815,0.011815,0.757576,64.121212,1.0,0.01163,4.076264,1.000000,0.757576,0.754677,0.878788
7758,"(Spring, تابان)",(MD.88),0.011815,0.015595,0.011815,1.000000,64.121212,1.0,0.01163,inf,0.996174,0.757576,1.000000,0.878788
7755,(MD.88),(تابان),0.015595,0.011815,0.011815,0.757576,64.121212,1.0,0.01163,4.076264,1.000000,0.757576,0.754677,0.878788
7763,(MD.88),"(Spring, تابان)",0.015595,0.011815,0.011815,0.757576,64.121212,1.0,0.01163,4.076264,1.000000,0.757576,0.754677,0.878788


In [150]:
late_rules = rules[rules['consequents'].apply(lambda x: 'Late' in x)].copy()
late_rules.shape

(2206, 14)

In [151]:
late_rules.sort_values(by='confidence', ascending=False).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
206,"(Evening, مشهد, MD83)","(Spring, Late)",0.016541,0.6569,0.015595,0.942857,1.435313,1.0,0.004730,6.004253,0.308389,0.023707,0.833451,0.483299
196,"(Spring, Evening, مشهد, MD83)",(Late),0.016541,0.6569,0.015595,0.942857,1.435313,1.0,0.004730,6.004253,0.308389,0.023707,0.833451,0.483299
186,"(Evening, مشهد, MD83)",(Late),0.016541,0.6569,0.015595,0.942857,1.435313,1.0,0.004730,6.004253,0.308389,0.023707,0.833451,0.483299
6900,"(Spring, آتا, MD83, Sunday)",(Late),0.015123,0.6569,0.014178,0.937500,1.427158,1.0,0.004243,5.489603,0.303903,0.021552,0.817837,0.479541
6911,"(آتا, MD83, Sunday)","(Spring, Late)",0.015123,0.6569,0.014178,0.937500,1.427158,1.0,0.004243,5.489603,0.303903,0.021552,0.817837,0.479541


In [152]:
late_rules_freeze = rules[rules['consequents'] == frozenset({'Late'})].copy()
late_rules_freeze.shape

(333, 14)

In [153]:
late_rules_freeze = late_rules_freeze[late_rules_freeze['antecedents'].apply(lambda x: 'Spring' not in x)].copy()
late_rules_freeze.shape

(166, 14)

In [154]:
late_rules_freeze.sort_values(by='confidence', ascending= False).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
186,"(Evening, مشهد, MD83)",(Late),0.016541,0.6569,0.015595,0.942857,1.435313,1.0,0.004730,6.004253,0.308389,0.023707,0.833451,0.483299
6889,"(آتا, MD83, Sunday)",(Late),0.015123,0.6569,0.014178,0.937500,1.427158,1.0,0.004243,5.489603,0.303903,0.021552,0.817837,0.479541
1757,"(Evening, MD83, آتا)",(Late),0.024102,0.6569,0.022212,0.921569,1.402906,1.0,0.006379,4.374527,0.294287,0.033716,0.771404,0.477691
2257,"(اهواز, MD83, آتا)",(Late),0.011342,0.6569,0.010397,0.916667,1.395444,1.0,0.002946,4.117202,0.286633,0.015805,0.757117,0.466247
1727,"(Evening, آتا)",(Late),0.034026,0.6569,0.031191,0.916667,1.395444,1.0,0.008839,4.117202,0.293364,0.047278,0.757117,0.482074


In [155]:
late_rules_freeze['antecedents_len'] = late_rules_freeze['antecedents'].apply(lambda x: len(x))

In [156]:
late_rules_freeze.sort_values(by=['antecedents_len'], ascending =[True]).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
9,(MD83),(Late),0.314272,0.6569,0.233459,0.742857,1.130853,1.0,0.027014,1.334279,0.168743,0.316464,0.250531,0.549126,1
45,(مشهد),(Late),0.226843,0.6569,0.166352,0.733333,1.116355,1.0,0.017338,1.286626,0.134808,0.231884,0.222773,0.493285,1
224,(Wednesday),(Late),0.198488,0.6569,0.138941,0.700000,1.065612,1.0,0.008555,1.143667,0.076819,0.193931,0.125620,0.455755,1
807,(كيش),(Late),0.083648,0.6569,0.055293,0.661017,1.006268,1.0,0.000344,1.012146,0.006797,0.080690,0.012000,0.372595,1
1059,(اطلس ایر),(Late),0.033554,0.6569,0.023629,0.704225,1.072044,1.0,0.001588,1.160005,0.069535,0.035436,0.137935,0.370098,1


## **By day**

In [157]:
days_of_the_week = set(df['scheduled_day_of_week'].unique())

def only_things(antecedents, _set):
    return antecedents.issubset(_set)

In [158]:
late_rules_by_day = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (days_of_the_week, ))].copy()
late_rules_by_day.shape
late_rules_by_day

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
224,(Wednesday),(Late),0.198488,0.6569,0.138941,0.700000,1.065612,1.0,0.008555,1.143667,0.076819,0.193931,0.125620,0.455755,1
8163,(Monday),(Late),0.140832,0.6569,0.095463,0.677852,1.031896,1.0,0.002951,1.065040,0.035977,0.135935,0.061068,0.411588,1
8717,(Tuesday),(Late),0.126181,0.6569,0.093573,0.741573,1.128898,1.0,0.010684,1.327649,0.130669,0.135709,0.246789,0.442010,1


## **By destination**

In [159]:
destinations = set(df['destination_or_origin'].unique())

In [160]:
late_rules_by_dest = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (destinations, ))].copy()
late_rules_by_dest.shape

(9, 15)

In [161]:
late_rules_by_dest.sort_values(by='confidence', ascending = False).head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
7169,(آبادان),(Late),0.022212,0.6569,0.017958,0.808511,1.230797,1.0,0.003368,1.791745,0.191778,0.027162,0.441885,0.417924,1
2149,(اهواز),(Late),0.068526,0.6569,0.054348,0.793103,1.207343,1.0,0.009333,1.658318,0.184369,0.080986,0.396979,0.437919,1
45,(مشهد),(Late),0.226843,0.6569,0.166352,0.733333,1.116355,1.0,0.017338,1.286626,0.134808,0.231884,0.222773,0.493285,1
4871,(بندرعباس),(Late),0.038280,0.6569,0.026465,0.691358,1.052456,1.0,0.001319,1.111645,0.051825,0.039576,0.100432,0.365823,1
6349,(تبریز),(Late),0.049622,0.6569,0.034026,0.685714,1.043864,1.0,0.001430,1.091682,0.044215,0.050597,0.083983,0.368756,1


## **by Airlines only**

In [162]:
airlines = set(df['airline'].unique())

In [163]:
late_rules_by_airline = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (airlines, ))].copy()
late_rules_by_airline.shape

(8, 15)

In [164]:
late_rules_by_airline

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
1059,(اطلس ایر),(Late),0.033554,0.6569,0.023629,0.704225,1.072044,1.0,0.001588,1.160005,0.069535,0.035436,0.137935,0.370098,1
1099,(آتا),(Late),0.152647,0.6569,0.125709,0.823529,1.253661,1.0,0.025435,1.944234,0.238786,0.183829,0.485659,0.507448,1
4073,(وارش),(Late),0.050095,0.6569,0.037807,0.754717,1.148907,1.0,0.004900,1.398793,0.136443,0.056497,0.285098,0.406135,1
4917,(آسمان),(Late),0.028828,0.6569,0.021267,0.737705,1.123010,1.0,0.002329,1.308069,0.112787,0.032006,0.235515,0.385040,1
5185,(کیش ایر),(Late),0.063327,0.6569,0.043478,0.686567,1.045163,1.0,0.001879,1.094653,0.046133,0.064246,0.086468,0.376377,1
5637,(قشم ایر),(Late),0.051985,0.6569,0.037335,0.718182,1.093290,1.0,0.003186,1.217452,0.090008,0.055595,0.178613,0.387508,1
5815,(ایران ایرتور),(Late),0.046314,0.6569,0.032609,0.704082,1.071825,1.0,0.002185,1.159442,0.070266,0.048626,0.137516,0.376861,1
7641,(زاگرس),(Late),0.037335,0.6569,0.033081,0.886076,1.348875,1.0,0.008556,3.011657,0.268672,0.050036,0.667957,0.468218,1


### **by Aircraft**

In [165]:
aircrafts = set(df['aircraft'].unique())

In [166]:
late_rules_by_aircraft = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (aircrafts, ))].copy()
late_rules_by_aircraft.shape

(6, 15)

In [167]:
late_rules_by_aircraft

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
9,(MD83),(Late),0.314272,0.6569,0.233459,0.742857,1.130853,1.0,0.027014,1.334279,0.168743,0.316464,0.250531,0.549126,1
3157,(rj100),(Late),0.025520,0.6569,0.017486,0.685185,1.043059,1.0,0.000722,1.089848,0.042362,0.026297,0.082441,0.355902,1
3741,(B737),(Late),0.092155,0.6569,0.066635,0.723077,1.100742,1.0,0.006099,1.238973,0.100812,0.097645,0.192880,0.412258,1
5045,(737),(Late),0.063800,0.6569,0.045841,0.718519,1.093802,1.0,0.003931,1.218909,0.091602,0.067927,0.179594,0.394151,1
7583,(MD82),(Late),0.039698,0.6569,0.027883,0.702381,1.069236,1.0,0.001805,1.152817,0.067430,0.041696,0.132559,0.372413,1
7747,(MD.88),(Late),0.015595,0.6569,0.011815,0.757576,1.153259,1.0,0.001570,1.415288,0.134998,0.017883,0.293430,0.387781,1


## **by Time**

In [168]:
times = set(df['scheduled_time_of_day'].unique())

In [169]:
late_rules_by_time = late_rules_freeze[late_rules_freeze['antecedents'].apply(only_things, args = (times, ))]
late_rules_by_time.shape

(1, 15)

In [170]:
late_rules_by_time

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_len
6943,(Evening),(Late),0.263233,0.6569,0.184783,0.701975,1.068618,1.0,0.011865,1.151246,0.087153,0.251285,0.131376,0.491635,1


Here are the key additional analyses you can provide to airlines:
1. Operational Performance Analytics

On-Time Performance (OTP) metrics by airline, route, and aircraft
Schedule reliability indicators
Aircraft utilization efficiency metrics

2. Delay Pattern Analysis

Time-based patterns: delays by hour, day of week, season
Holiday impact analysis: performance during special periods
Weekend vs weekday comparison

3. Predictive Delay Risk Scoring

Risk assessment for different operational scenarios
Probability-based scoring for delay likelihood
Early warning systems for high-risk flights

4. Resource Optimization

Gate/counter utilization analysis
Aircraft rotation efficiency
Peak hour capacity planning

5. Cost Impact Analysis

Financial impact of delays (estimated costs)
Monthly cost trends
ROI analysis for operational improvements

6. Competitive Benchmarking

Industry comparison metrics
Performance gaps identification
Market positioning analysis

7. Route Network Analysis

Route profitability assessment
Network optimization recommendations
Problematic routes identification

8. Passenger Impact Analysis

Severe delay analysis (>30 minutes)
Passenger disruption patterns
Service quality metrics

Key Benefits for Airlines:
Strategic Planning: Use seasonal and time-based patterns for capacity planning
Operational Efficiency: Identify bottlenecks and optimization opportunities
Cost Management: Quantify delay costs and prioritize improvements
Competitive Advantage: Benchmark against industry standards
Customer Experience: Reduce passenger disruptions and improve satisfaction
Risk Management: Proactive identification of high-risk scenarios
These analyses complement your association rule mining by providing quantitative metrics, predictive insights, and actionable recommendations that airlines can use for operational improvements and strategic decision-making.RetryClaude can make mistakes. Please double-check responses. Sonnet 4

## On-Time Performance Analysis

In [171]:
# Overall airline performance metrics
performance_metrics = air.groupby('airline').agg({
    'delay_minutes': ['mean', 'median', 'std', 'count'],
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100
}).round(2)

performance_metrics.columns = ['avg_delay', 'median_delay', 'delay_std', 'flight_count', 'ontime_percentage']
performance_metrics.sort_values('ontime_percentage', ascending=False)

,avg_delay,median_delay,delay_std,flight_count,ontime_percentage
airline,,,,,
آتا,52.98,32.0,55.79,323,0.0
آساجت,20.07,15.0,23.17,43,0.0
آسمان,53.69,22.0,82.90,61,0.0
آوا,41.58,20.0,47.69,24,0.0
اروان,26.18,20.0,15.29,11,0.0
اطلس ایر,52.30,32.0,73.56,71,0.0
ایران ایر,35.03,21.5,53.12,122,0.0
ایران ایرتور,44.63,23.0,60.24,98,0.0
تابان,42.08,38.0,38.47,25,0.0


In [172]:
# Performance by destination
dest_performance = air.groupby('destination_or_origin').agg({
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100
}).round(2)
dest_performance.columns = ['avg_delay', 'late_percentage']
dest_performance.sort_values('late_percentage', ascending=False).head(10)

,avg_delay,late_percentage
destination_or_origin,,
زنجان,46.00,100.00
جاسک,25.75,100.00
زاهدان/چابهار,40.33,100.00
سبزوار,24.00,100.00
نوشهر,17.00,100.00
جیرفت,22.17,83.33
خارگ,28.50,83.33
سراوان,30.00,83.33
آبادان,50.89,80.85


In [173]:
# Performance by aircraft type
aircraft_performance = air.groupby('aircraft').agg({
    'delay_minutes': ['mean', 'count'],
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100
}).round(2)
aircraft_performance.columns = ['avg_delay', 'flight_count', 'ontime_percentage']
aircraft_performance[aircraft_performance['flight_count'] >= 10]  # Filter for statistical significance

,avg_delay,flight_count,ontime_percentage
aircraft,,,
146-200,19.96,98,0.0
737,43.27,135,0.0
737-700,27.57,144,0.0
747,26.00,19,0.0
A319,35.85,39,0.0
A320,14.60,10,0.0
A321,23.00,16,0.0
B737,42.09,195,0.0
CRJ200,28.71,34,0.0


## Temporal Pattern Analysis

In [174]:
# Peak delay hours analysis
hourly_delays = air.groupby('Scheduled_Hour_of_Day').agg({
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100
}).round(2)
hourly_delays.columns = ['avg_delay', 'late_percentage']
hourly_delays.sort_values('late_percentage', ascending=False)

,avg_delay,late_percentage
Scheduled_Hour_of_Day,,
1,75.50,100.00
2,27.67,100.00
23,25.00,75.00
15,40.36,75.00
4,49.33,72.22
20,39.79,71.43
19,40.00,70.90
17,44.66,70.59
11,41.50,70.59


In [175]:
# Weekly pattern analysis
weekly_pattern = air.groupby('scheduled_day_of_week').agg({
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100,
    'flight_number': 'count'
}).round(2)
weekly_pattern.columns = ['avg_delay', 'late_percentage', 'flight_volume']
weekly_pattern

,avg_delay,late_percentage,flight_volume
scheduled_day_of_week,,,
Friday,29.12,53.47,144
Monday,36.73,67.79,298
Saturday,28.96,61.31,305
Sunday,35.45,63.10,290
Thursday,27.16,63.52,392
Tuesday,44.21,74.16,267
Wednesday,43.09,70.00,420


In [176]:
# Seasonal impact analysis
seasonal_analysis = air.groupby('scheduled_season').agg({
    'delay_minutes': ['mean', 'median'],
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100
}).round(2)
seasonal_analysis.columns = ['avg_delay', 'median_delay', 'late_percentage']
seasonal_analysis

,avg_delay,median_delay,late_percentage
scheduled_season,,,
Spring,35.35,22.0,65.69


## Operational Efficiency Analysis

In [177]:
# Counter utilization and efficiency
counter_analysis = air.groupby('counter').agg({
    'delay_minutes': 'mean',
    'flight_number': 'count',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100
}).round(2)
counter_analysis.columns = ['avg_delay', 'flight_count', 'ontime_percentage']
counter_analysis.sort_values('flight_count', ascending=False).head(10)

,avg_delay,flight_count,ontime_percentage
counter,,,
"18, 19",35.53,139,0.0
"12, 13",46.91,125,0.0
"9, 10",40.69,121,0.0
3,32.16,104,0.0
"14, 15",26.84,91,0.0
"16, 17",34.63,86,0.0
"24, 25",54.40,73,0.0
"26, 27",52.42,72,0.0
"6, 7",28.27,70,0.0


In [178]:
# Holiday impact analysis
holiday_impact = air.groupby(['Normal_holiday', 'is_weekend']).agg({
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100
}).round(2)
holiday_impact.columns = ['avg_delay', 'late_percentage']
holiday_impact

avg_delay  late_percentage
Normal_holiday is_weekend                            
0              0               37.79            67.24
               1               27.69            60.82
1              0               39.85            68.55

In [179]:
# Flight frequency vs delay correlation
airline_freq_delay = air.groupby('airline').agg({
    'flight_number': 'count',
    'delay_minutes': 'mean'
}).round(2)
airline_freq_delay.columns = ['flight_frequency', 'avg_delay']
airline_freq_delay['efficiency_score'] = airline_freq_delay['flight_frequency'] / (airline_freq_delay['avg_delay'] + 1)
airline_freq_delay.sort_values('efficiency_score', ascending=False)

,flight_frequency,avg_delay,efficiency_score
airline,,,
ماهان,221,18.43,11.374164
آتا,323,52.98,5.983698
کاسپین,141,31.45,4.345146
کارون,124,31.98,3.759854
قشم ایر,110,28.89,3.680161
چابهار,87,24.06,3.471668
رایمون,43,11.58,3.418124
ایران ایر,122,35.03,3.386067
کیش ایر,134,39.54,3.305377


## Risk Assessment and Predictive Analysis

In [180]:
# High-risk combinations (airline + destination + time)
risk_combinations = air.groupby(['airline', 'destination_or_origin', 'scheduled_time_of_day']).agg({
    'delay_minutes': ['mean', 'count'],
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100
}).round(2)
risk_combinations.columns = ['avg_delay', 'flight_count', 'late_percentage']
high_risk = risk_combinations[(risk_combinations['flight_count'] >= 5) & (risk_combinations['late_percentage'] > 50)]
high_risk.sort_values('late_percentage', ascending=False)

avg_delay  \
airline      destination_or_origin scheduled_time_of_day              
آتا          آبادان                Evening                    88.17   
             ارومیه                Evening                    95.38   
             تبریز                 Afternoon                  70.00   
             اهواز                 Evening                    74.89   
آسمان        مشهد                  Evening                    84.40   
...                                                             ...   
ماهان        کرمان                 Evening                    26.88   
ایران ایرتور بندرعباس              Morning                    18.89   
آتا          یزد                   Night                      32.67   
رایمون       رشت                   Evening                    24.67   
ساها         شیراز                 Evening                    21.27   

                                                          flight_count  \
airline      destination_or_origin scheduled_time_of_day                 
آتا          آبادان                Evening                           6   
             ارومیه                Evening                           8   
             تبریز                 Afternoon                         8   
             اهواز                 Evening                           9   
آسمان        مشهد                  Evening                           5   
...                                                                ...   
ماهان        کرمان                 Evening                          16   
ایران ایرتور بندرعباس              Morning                           9   
آتا          یزد                   Night                             9   
رایمون       رشت                   Evening                           9   
ساها         شیراز                 Evening                          15   

                                                          late_percentage  
airline      destination_or_origin scheduled_time_of_day                   
آتا          آبادان                Evening                         100.00  
             ارومیه                Evening                         100.00  
             تبریز                 Afternoon                       100.00  
             اهواز                 Evening                         100.00  
آسمان        مشهد                  Evening                         100.00  
...                                                                   ...  
ماهان        کرمان                 Evening                          56.25  
ایران ایرتور بندرعباس              Morning                          55.56  
آتا          یزد                   Night                            55.56  
رایمون       رشت                   Evening                          55.56  
ساها         شیراز                 Evening                          53.33  

[108 rows x 3 columns]

In [181]:
# Delay propagation analysis (same day consecutive flights)
air_sorted = air.sort_values(['airline', 'scheduled_datetime'])
air_sorted['prev_delay'] = air_sorted.groupby('airline')['delay_minutes'].shift(1)
delay_propagation = air_sorted.groupby(pd.cut(air_sorted['prev_delay'], bins=[-1, 0, 30, 60, float('inf')], labels=['OnTime', 'Short', 'Medium', 'Long']))['delay_minutes'].mean()
delay_propagation

prev_delay
OnTime    27.444444
Short     30.035372
Medium    39.262136
Long      57.049470
Name: delay_minutes, dtype: float64

In [182]:
# Weather/seasonal delay patterns
weather_proxy = air.groupby(['scheduled_season', 'scheduled_time_of_day']).agg({
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100
}).round(2)
weather_proxy.columns = ['avg_delay', 'late_percentage']
weather_proxy.sort_values('late_percentage', ascending=False)

avg_delay  late_percentage
scheduled_season scheduled_time_of_day                            
Spring           Evening                    40.55            70.20
                 Afternoon                  38.31            65.67
                 Night                      35.48            65.57
                 Morning                    31.11            63.08

## Network and Route Analysis

In [183]:
# Route efficiency analysis
route_efficiency = air.groupby('destination_or_origin').agg({
    'delay_minutes': ['mean', 'std', 'count'],
    'airline': 'nunique',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100
}).round(2)
route_efficiency.columns = ['avg_delay', 'delay_variability', 'flight_frequency', 'competing_airlines', 'ontime_percentage']
route_efficiency['route_score'] = route_efficiency['ontime_percentage'] / (route_efficiency['delay_variability'] + 1)
route_efficiency.sort_values('route_score', ascending=False)

,avg_delay,delay_variability,flight_frequency,competing_airlines,ontime_percentage,route_score
destination_or_origin,,,,,,
آبادان,50.89,55.41,47,10,0.0,0.0
آغاجاری,25.20,54.34,5,1,0.0,0.0
اراک,11.50,19.09,2,2,0.0,0.0
اردبیل,27.47,49.91,32,4,0.0,0.0
ارومیه,34.27,41.69,56,9,0.0,0.0
اصفهان,19.63,17.59,46,11,0.0,0.0
اصفهان/بهرگان,20.67,7.37,3,1,0.0,0.0
اهواز,39.32,38.41,145,16,0.0,0.0
ایرانشهر,21.60,10.60,5,1,0.0,0.0


In [184]:
# Airline market share by destination
market_share = air.groupby(['destination_or_origin', 'airline']).size().unstack(fill_value=0)
market_share_pct = market_share.div(market_share.sum(axis=1), axis=0) * 100
market_share_pct.round(2)

airline,آتا,آساجت,آسمان,آوا,اروان,اطلس ایر,ایران ایر,ایران ایرتور,تابان,رایمون,...,ماهان,معراج,وارش,پارس ایر,پويا,چابهار,کارون,کاسپین,کیش ایر,یزد ایر
destination_or_origin,,,,,,,,,,,,,,,,,,,,,
آبادان,19.15,10.64,6.38,0.00,0.00,8.51,12.77,0.00,0.00,0.00,...,0.00,0.00,0.00,14.89,0.00,8.51,0.00,0.00,10.64,0.00
آغاجاری,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,0.00,0.00
اراک,0.00,50.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,50.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
اردبیل,15.62,0.00,6.25,0.00,0.00,0.00,9.38,0.00,0.00,0.00,...,68.75,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
ارومیه,33.93,0.00,1.79,0.00,0.00,0.00,8.93,12.50,0.00,0.00,...,14.29,0.00,0.00,0.00,0.00,0.00,7.14,12.50,0.00,5.36
اصفهان,0.00,10.87,2.17,0.00,0.00,2.17,15.22,0.00,0.00,26.09,...,0.00,0.00,0.00,4.35,2.17,0.00,10.87,0.00,6.52,6.52
اصفهان/بهرگان,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00
اهواز,28.97,0.00,2.07,0.00,0.00,0.69,4.83,3.45,0.00,0.00,...,0.00,0.00,8.28,2.07,0.00,4.14,17.24,0.00,2.07,2.07
ایرانشهر,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,100.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [185]:
# Aircraft utilization patterns
aircraft_utilization = air.groupby(['aircraft', 'airline']).agg({
    'flight_number': 'count',
    'delay_minutes': 'mean',
    'destination_or_origin': 'nunique'
}).round(2)
aircraft_utilization.columns = ['total_flights', 'avg_delay', 'routes_served']
aircraft_utilization['utilization_efficiency'] = aircraft_utilization['total_flights'] / (aircraft_utilization['avg_delay'] + 1)
aircraft_utilization.sort_values('utilization_efficiency', ascending=False)

,,total_flights,avg_delay,routes_served,utilization_efficiency
aircraft,airline,,,,
RJ85,ماهان,74,16.08,22,4.332553
146-200,ماهان,74,17.53,22,3.993524
MD83,آتا,210,55.36,13,3.726047
EMB145,رایمون,43,11.58,6,3.418124
MD83,چابهار,75,24.00,10,3.000000
...,...,...,...,...,...
737-700,آوا,2,80.00,2,0.024691
CITATION,ماهان,1,46.00,1,0.021277
AB3,ایران ایر,1,86.00,1,0.011494


## Competitive Intelligence

In [186]:
# Head-to-head airline comparison on same routes
route_competition = air.groupby(['destination_or_origin', 'airline']).agg({
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100,
    'flight_number': 'count'
}).round(2)
route_competition.columns = ['avg_delay', 'ontime_percentage', 'flight_count']

# Get routes with multiple airlines
competitive_routes = route_competition.groupby('destination_or_origin').filter(lambda x: len(x) > 1)
competitive_routes.sort_values(['destination_or_origin', 'ontime_percentage'], ascending=[True, False])

avg_delay  ontime_percentage  flight_count
destination_or_origin airline                                              
آبادان                آتا            79.22                0.0             9
                      آساجت          57.00                0.0             5
                      آسمان         132.67                0.0             3
                      اطلس ایر       41.25                0.0             4
                      ایران ایر      44.00                0.0             6
...                                    ...                ...           ...
یزد                   قدر ایر        17.00                0.0             2
                      پويا           11.60                0.0             5
                      کارون          19.40                0.0            10
                      کیش ایر        18.33                0.0             6
                      یزد ایر        25.66                0.0            29

[258 rows x 3 columns]

In [187]:
# Airline performance benchmarking
airline_benchmark = air.groupby('airline').agg({
    'delay_minutes': ['mean', 'median', 'std'],
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100,
    'destination_or_origin': 'nunique',
    'flight_number': 'count'
}).round(2)
airline_benchmark.columns = ['avg_delay', 'median_delay', 'delay_consistency', 'ontime_percentage', 'routes_served', 'total_flights']
airline_benchmark['overall_score'] = (airline_benchmark['ontime_percentage'] * 0.4 + 
                                    (100 - airline_benchmark['avg_delay']) * 0.3 + 
                                    (100 - airline_benchmark['delay_consistency']) * 0.3)
airline_benchmark.sort_values('overall_score', ascending=False)

,avg_delay,median_delay,delay_consistency,ontime_percentage,routes_served,total_flights,overall_score
airline,,,,,,,
رایمون,11.58,9.0,16.00,0.0,6,43,51.726
پويا,17.72,17.0,12.42,0.0,11,43,50.958
معراج,20.08,16.0,13.75,0.0,6,24,49.851
ماهان,18.43,15.0,16.44,0.0,27,221,49.539
فلای پرشیا,19.04,17.0,20.08,0.0,5,45,48.264
یزد ایر,22.49,17.0,17.40,0.0,8,49,48.033
اروان,26.18,20.0,15.29,0.0,1,11,47.559
طوس ایر,28.20,33.0,13.99,0.0,3,5,47.343
آساجت,20.07,15.0,23.17,0.0,8,43,47.028


## Advanced Cost Impact Analysis

In [188]:
# Estimate financial impact of delays (using industry averages)
COST_PER_MINUTE = 50  # USD per minute of delay (industry average)
PASSENGER_COMPENSATION_THRESHOLD = 120  # minutes

air['estimated_cost'] = air['delay_minutes'] * COST_PER_MINUTE
air['compensation_required'] = air['delay_minutes'] > PASSENGER_COMPENSATION_THRESHOLD

cost_analysis = air.groupby('airline').agg({
    'estimated_cost': ['sum', 'mean'],
    'compensation_required': 'sum',
    'delay_minutes': 'count'
}).round(2)
cost_analysis.columns = ['total_cost_estimate', 'avg_cost_per_flight', 'compensation_cases', 'total_flights']
cost_analysis['cost_per_flight'] = cost_analysis['total_cost_estimate'] / cost_analysis['total_flights']
cost_analysis.sort_values('total_cost_estimate', ascending=False)

,total_cost_estimate,avg_cost_per_flight,compensation_cases,total_flights,cost_per_flight
airline,,,,,
آتا,855700.0,2649.23,35,323,2649.226006
کیش ایر,264900.0,1976.87,7,134,1976.865672
وارش,262800.0,2479.25,8,106,2479.245283
کاسپین,221750.0,1572.70,5,141,1572.695035
ایران ایرتور,218700.0,2231.63,10,98,2231.632653
ایران ایر,213700.0,1751.64,7,122,1751.639344
ماهان,203650.0,921.49,2,221,921.493213
کارون,198300.0,1599.19,5,124,1599.193548
اطلس ایر,185650.0,2614.79,7,71,2614.788732


In [189]:
# Monthly cost trends
air['month'] = pd.to_datetime(air['scheduled_datetime']).dt.month
monthly_costs = air.groupby(['month', 'airline']).agg({
    'estimated_cost': 'sum',
    'delay_minutes': 'mean'
}).round(2)
monthly_costs.columns = ['monthly_cost', 'avg_delay']
monthly_costs.reset_index().pivot(index='month', columns='airline', values='monthly_cost')

airline,آتا,آساجت,آسمان,آوا,اروان,اطلس ایر,ایران ایر,ایران ایرتور,تابان,رایمون,...,ماهان,معراج,وارش,پارس ایر,پويا,چابهار,کارون,کاسپین,کیش ایر,یزد ایر
month,,,,,,,,,,,,,,,,,,,,,
5,321850.0,13000.0,61250.0,22550.0,8050.0,55450.0,100250.0,69700.0,33200.0,8800.0,...,76750.0,9350.0,88600.0,33500.0,14350.0,56950.0,87950.0,65050.0,67850.0,27450.0
6,533850.0,30150.0,102500.0,27350.0,6350.0,130200.0,113450.0,149000.0,19400.0,16100.0,...,126900.0,14750.0,174200.0,103350.0,23750.0,47700.0,110350.0,156700.0,197050.0,27650.0


## Severe Delay Analysis (Passenger Impact)

In [190]:
# Severe delay categorization
def categorize_delay(minutes):
    if minutes <= 15:
        return 'Acceptable'
    elif minutes <= 30:
        return 'Minor'
    elif minutes <= 60:
        return 'Moderate'
    elif minutes <= 120:
        return 'Severe'
    else:
        return 'Critical'

air['delay_category'] = air['delay_minutes'].apply(categorize_delay)

# Passenger impact analysis
passenger_impact = air.groupby(['airline', 'delay_category']).size().unstack(fill_value=0)
passenger_impact_pct = passenger_impact.div(passenger_impact.sum(axis=1), axis=0) * 100
passenger_impact_pct.round(2)

delay_category,Acceptable,Critical,Minor,Moderate,Severe
airline,,,,,
آتا,17.65,10.84,30.34,23.84,17.34
آساجت,53.49,0.00,27.91,13.95,4.65
آسمان,26.23,14.75,39.34,13.11,6.56
آوا,33.33,16.67,33.33,12.50,4.17
اروان,27.27,0.00,45.45,18.18,9.09
اطلس ایر,29.58,9.86,18.31,32.39,9.86
ایران ایر,36.07,5.74,38.52,12.30,7.38
ایران ایرتور,29.59,10.20,36.73,15.31,8.16
تابان,20.00,4.00,20.00,44.00,12.00


In [191]:
# Critical delay incidents (>2 hours)
critical_delays = air[air['delay_minutes'] > 120].copy()
critical_analysis = critical_delays.groupby(['airline', 'destination_or_origin']).agg({
    'delay_minutes': ['count', 'mean'],
    'flight_number': 'count'
}).round(2)
critical_analysis.columns = ['critical_incidents', 'avg_critical_delay', 'affected_flights']
critical_analysis.sort_values('critical_incidents', ascending=False)

critical_incidents  avg_critical_delay  \
airline  destination_or_origin                                           
آتا      مشهد                                    9              187.11   
         تبریز                                   5              160.00   
         شیراز                                   5              202.40   
         اهواز                                   4              163.00   
اطلس ایر مشهد                                    4              296.50   
...                                            ...                 ...   
کاسپین   عسلویه                                  1              227.00   
         لارستان                                 1              220.00   
         قشم                                     1              123.00   
کیش ایر  شیراز                                   1              144.00   
         بندرعباس                                1              176.00   

                                affected_flights  
airline  destination_or_origin                    
آتا      مشهد                                  9  
         تبریز                                 5  
         شیراز                                 5  
         اهواز                                 4  
اطلس ایر مشهد                                  4  
...                                          ...  
کاسپین   عسلویه                                1  
         لارستان                               1  
         قشم                                   1  
کیش ایر  شیراز                                 1  
         بندرعباس                              1  

[64 rows x 3 columns]

## Operational Bottleneck Analysis

In [192]:
# Counter/Gate bottleneck analysis
counter_bottleneck = air.groupby('counter').agg({
    'delay_minutes': ['mean', 'std', 'count'],
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100
}).round(2)
counter_bottleneck.columns = ['avg_delay', 'delay_variability', 'flight_volume', 'late_percentage']
counter_bottleneck['bottleneck_score'] = (counter_bottleneck['late_percentage'] * 
                                         counter_bottleneck['delay_variability'] / 100)
counter_bottleneck.sort_values('bottleneck_score', ascending=False).head(10)

,avg_delay,delay_variability,flight_volume,late_percentage,bottleneck_score
counter,,,,,
12,64.80,128.40,10,70.00,89.880000
10,66.00,85.39,7,85.71,73.187769
29,50.83,82.83,12,75.00,62.122500
28,57.56,81.33,18,72.22,58.736526
"28, 29",62.32,67.55,60,86.67,58.545585
7,42.63,67.11,43,79.07,53.063877
5,38.19,66.94,67,79.10,52.949540
27,48.50,58.33,8,87.50,51.038750
"26, 27",52.42,57.28,72,87.50,50.120000


In [193]:
# Peak hour capacity analysis
peak_analysis = air.groupby('Scheduled_Hour_of_Day').agg({
    'flight_number': 'count',
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100
}).round(2)
peak_analysis.columns = ['flight_volume', 'avg_delay', 'late_percentage']
peak_analysis['capacity_stress'] = peak_analysis['flight_volume'] * peak_analysis['late_percentage'] / 100
peak_analysis.sort_values('capacity_stress', ascending=False)

,flight_volume,avg_delay,late_percentage,capacity_stress
Scheduled_Hour_of_Day,,,,
6,230,29.26,66.96,154.0080
5,214,31.06,59.81,127.9934
18,161,38.15,68.32,109.9952
7,148,24.81,68.92,102.0016
17,136,44.66,70.59,96.0024
19,134,40.00,70.90,95.0060
20,126,39.79,71.43,90.0018
16,107,47.08,69.16,74.0012
21,93,36.82,69.89,64.9977


## Predictive Delay Risk Scoring

In [194]:
# Risk scoring based on multiple factors
def calculate_risk_score(row):
    score = 0
    
    # Time-based risk
    if row['Scheduled_Hour_of_Day'] in [6, 7, 8, 18, 19, 20]:  # Peak hours
        score += 30
    
    # Day-based risk
    if row['scheduled_day_of_week'] in ['Friday', 'Saturday']:
        score += 20
    
    # Holiday risk
    if row['Normal_holiday'] == 1:
        score += 25
    
    # Seasonal risk
    if row['scheduled_season'] == 'Winter':
        score += 15
    
    return score

air['risk_score'] = air.apply(calculate_risk_score, axis=1)

# Risk validation
risk_validation = air.groupby(pd.cut(air['risk_score'], bins=[0, 25, 50, 75, 100], labels=['Low', 'Medium', 'High', 'Very High'])).agg({
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100,
    'delay_minutes': 'mean'
}).round(2)
risk_validation.columns = ['actual_late_percentage', 'actual_avg_delay']
risk_validation

,actual_late_percentage,actual_avg_delay
risk_score,,
Low,60.59,31.16
Medium,66.99,32.59
High,74.51,42.27
Very High,NaN,NaN


In [195]:
# Airline-specific risk profiles
airline_risk = air.groupby('airline').agg({
    'risk_score': 'mean',
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'Late').sum() / len(x) * 100
}).round(2)
airline_risk.columns = ['avg_risk_score', 'avg_delay', 'late_percentage']
airline_risk['risk_accuracy'] = abs(airline_risk['avg_risk_score'] - airline_risk['late_percentage'])
airline_risk.sort_values('avg_risk_score', ascending=False)

,avg_risk_score,avg_delay,late_percentage,risk_accuracy
airline,,,,
اروان,37.73,26.18,72.73,35.00
قدر ایر,29.44,26.89,55.56,26.12
ساها,26.82,26.11,56.82,30.00
تابان,26.40,42.08,80.00,53.60
طوس ایر,26.00,28.20,80.00,54.00
معراج,25.00,20.08,50.00,25.00
ماهان,22.35,18.43,47.96,25.61
آوا,21.88,41.58,66.67,44.79
ایران ایرتور,21.02,44.63,70.41,49.39


## Schedule Optimization Analysis

In [196]:
# Optimal scheduling windows
schedule_optimization = air.groupby(['scheduled_day_of_week', 'Scheduled_Hour_of_Day']).agg({
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100,
    'flight_number': 'count'
}).round(2)
schedule_optimization.columns = ['avg_delay', 'ontime_percentage', 'flight_count']

# Find optimal time slots (low delay, high on-time performance)
optimal_slots = schedule_optimization[
    (schedule_optimization['avg_delay'] < schedule_optimization['avg_delay'].median()) &
    (schedule_optimization['ontime_percentage'] > schedule_optimization['ontime_percentage'].median())
]
optimal_slots.sort_values('ontime_percentage', ascending=False)

,,avg_delay,ontime_percentage,flight_count
scheduled_day_of_week,Scheduled_Hour_of_Day,,,


In [197]:
# Buffer time analysis
air['buffer_effectiveness'] = air['delay_minutes'] < 15  # Within acceptable range
buffer_analysis = air.groupby(['airline', 'scheduled_time_of_day']).agg({
    'buffer_effectiveness': lambda x: x.sum() / len(x) * 100,
    'delay_minutes': ['mean', 'std']
}).round(2)
buffer_analysis.columns = ['buffer_success_rate', 'avg_delay', 'delay_variability']
buffer_analysis.sort_values('buffer_success_rate', ascending=False)

,,buffer_success_rate,avg_delay,delay_variability
airline,scheduled_time_of_day,,,
آساجت,Night,100.00,-2.00,NaN
رایمون,Night,100.00,1.40,5.98
طوس ایر,Evening,100.00,4.00,NaN
پويا,Night,100.00,9.00,NaN
رایمون,Morning,81.82,8.18,6.72
...,...,...,...,...
طوس ایر,Night,0.00,37.50,0.71
زاگرس,Afternoon,0.00,20.50,6.36
وارش,Afternoon,0.00,45.75,28.39


## Quality Control and Anomaly Detection

In [198]:
# Anomaly detection for unusual delays
from scipy import stats

# Calculate z-scores for delay times
air['delay_zscore'] = stats.zscore(air['delay_minutes'])
anomalies = air[abs(air['delay_zscore']) > 2]  # More than 2 standard deviations

anomaly_analysis = anomalies.groupby(['airline', 'destination_or_origin']).agg({
    'delay_minutes': ['count', 'mean'],
    'flight_number': 'count'
}).round(2)
anomaly_analysis.columns = ['anomaly_count', 'avg_anomaly_delay', 'total_flights']
anomaly_analysis['anomaly_rate'] = anomaly_analysis['anomaly_count'] / anomaly_analysis['total_flights'] * 100
anomaly_analysis.sort_values('anomaly_rate', ascending=False)

anomaly_count  avg_anomaly_delay  \
airline      destination_or_origin                                     
آتا          آبادان                             2             167.50   
             ارومیه                             2             183.50   
             اهواز                              3             176.33   
             تبریز                              4             169.50   
             سیرجان                             1             187.00   
             شیراز                              4             221.50   
             كيش                                2             222.50   
             مشهد                               7             204.57   
             کرمان                              3             218.67   
             یزد                                1             142.00   
آسمان        آبادان                             1             313.00   
             اردبیل                             1             284.00   
             تبریز                              1             422.00   
             شیراز                              1             263.00   
             مشهد                               1             208.00   
             گرگان                              1             149.00   
آوا          بندرعباس                           1             147.00   
             مشهد                               2             145.50   
اطلس ایر     قشم                                1             176.00   
             كيش                                2             162.00   
             مشهد                               4             296.50   
ایران ایر    بجنورد                             1             179.00   
             عسلویه                             1             137.00   
             كيش                                1             318.00   
             لارستان                            1             386.00   
             همدان                              1             229.00   
ایران ایرتور بندرعباس                           2             179.50   
             بوشهر                              1             182.00   
             تبریز                              1             148.00   
             زاهدان                             1             199.00   
             كيش                                1             273.00   
             مشهد                               2             262.50   
تابان        مشهد                               1             195.00   
زاگرس        مشهد                               1             196.00   
ساها         مشهد                               1             280.00   
ماهان        زاهدان                             1             136.00   
             کرمان                              1             132.00   
وارش         اهواز                              1             148.00   
             بندرعباس                           1             138.00   
             بوشهر                              1             155.00   
             ساری                               1             158.00   
             مشهد                               3             376.33   
پارس ایر     كيش                                1             174.00   
             مشهد                               1             435.00   
کارون        اهواز                              1             221.00   
             دزفول                              1             165.00   
             سیری                               1             133.00   
             لاوان                              1             161.00   
کاسپین       عسلویه                             1             227.00   
             كيش                                2             213.50   
             لارستان                            1             220.00   
کیش ایر      بندرعباس                           1             176.00   
             شیراز                              1             144.00   
             عسلویه                             2             267.00   
    

In [199]:
# Consistency scoring
consistency_analysis = air.groupby('airline').agg({
    'delay_minutes': ['std', 'mean'],
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100
}).round(2)
consistency_analysis.columns = ['delay_std', 'avg_delay', 'ontime_percentage']
consistency_analysis['consistency_score'] = (100 - consistency_analysis['delay_std']) * 0.6 + consistency_analysis['ontime_percentage'] * 0.4
consistency_analysis.sort_values('consistency_score', ascending=False)

,delay_std,avg_delay,ontime_percentage,consistency_score
airline,,,,
پويا,12.42,17.72,0.0,52.548
معراج,13.75,20.08,0.0,51.750
طوس ایر,13.99,28.20,0.0,51.606
اروان,15.29,26.18,0.0,50.826
رایمون,16.00,11.58,0.0,50.400
ماهان,16.44,18.43,0.0,50.136
یزد ایر,17.40,22.49,0.0,49.560
قدر ایر,17.95,26.89,0.0,49.230
فلای پرشیا,20.08,19.04,0.0,47.952


## Fleet Management Insights

In [200]:
# Aircraft efficiency analysis
aircraft_efficiency = air.groupby(['aircraft', 'airline']).agg({
    'delay_minutes': ['mean', 'std', 'count'],
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100,
    'destination_or_origin': 'nunique'
}).round(2)
aircraft_efficiency.columns = ['avg_delay', 'delay_consistency', 'flight_count', 'ontime_percentage', 'routes_served']
aircraft_efficiency['efficiency_score'] = (aircraft_efficiency['ontime_percentage'] / 
                                         (aircraft_efficiency['avg_delay'] + 1))
aircraft_efficiency[aircraft_efficiency['flight_count'] >= 10].sort_values('efficiency_score', ascending=False)

avg_delay  delay_consistency  flight_count  \
aircraft airline                                                    
146-200  ماهان             17.53              17.87            74   
         یزد ایر           27.46              17.90            24   
737      آتا               65.36              72.17            44   
         پارس ایر          59.12              99.33            17   
         کارون             26.16              20.88            43   
         کاسپین            22.28              14.90            25   
737-700  اطلس ایر          28.78              25.66            18   
         ساها              26.29              41.85            42   
         فلای پرشیا        14.33              13.27            27   
         وارش              45.85              34.09            13   
         کارون             24.79              16.92            14   
747      ماهان             26.00              11.38            19   
A319     ایران ایر         35.85              63.57            39   
A321     معراج             23.00              15.06            16   
B737     آتا               38.95              33.73            21   
         آوا               27.89              28.44            19   
         فلای پرشیا        26.11              26.20            18   
         وارش              50.11              76.24            93   
         پارس ایر          40.13              31.71            15   
         کارون             46.45              66.03            11   
CRJ200   پارس ایر          28.76              40.65            33   
EMB145   آساجت             20.07              23.17            43   
         رایمون            11.58              16.00            43   
         پويا              17.72              12.42            43   
F100     آسمان             53.69              82.90            61   
         ایران ایر         31.16              43.31            57   
         قشم ایر           24.47              16.76            62   
         کارون             35.41              41.11            56   
MD.88    تابان             42.08              38.47            25   
MD82     آتا               45.23              36.00            13   
         ایران ایرتور      35.50              29.85            12   
         زاگرس             32.10              20.06            10   
         چابهار            24.42              14.04            12   
         کیش ایر           27.77              18.47            31   
MD83     آتا               55.36              56.84           210   
         اطلس ایر          58.14              65.23            21   
         ایران ایرتور      45.91              63.34            86   
         زاگرس             32.77              24.52            69   
         چابهار            24.00              23.31            75   
         کاسپین            33.27              40.33            98   
         کیش ایر           43.08              67.11           103   
RJ100    قشم ایر           24.70              10.85            20   
         ماهان             23.82              36.34            11   
RJ85     اطلس ایر          66.63             109.10            19   
         ماهان             16.08              13.37            74   
         یزد ایر           16.00              11.69            20   
md88     آتا               37.27              32.06            11   
rj100    قشم ایر           45.33              30.27            24   
         ماهان             17.35               8.21            26   

                       ontime_percentage  routes_served  efficiency_score  
aircraft airline                                                           
146-200  ماهان                       0.0             22               0.0  
         یزد ایر                     0.0              7               0.0  
737      آتا                         0.0             10               0.0  
         پارس ایر                    0.0              5               0.0  
         کارون                       0.0       

In [201]:
# Aircraft age proxy analysis (using registration patterns)
air['aircraft_age_proxy'] = air['register'].str.extract('(\d+)').astype(float, errors='ignore')
age_performance = air.groupby('aircraft_age_proxy').agg({
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100
}).round(2)
age_performance.columns = ['avg_delay', 'ontime_percentage']
age_performance.sort_values('avg_delay')

,avg_delay,ontime_percentage
aircraft_age_proxy,,
152282.0,13.5,0.0
152256.0,26.0,0.0
152286.0,45.5,0.0


## Customer Experience Metrics

In [202]:
# Service quality scoring
def calculate_service_score(row):
    score = 100
    
    # Delay penalty
    if row['delay_minutes'] > 60:
        score -= 50
    elif row['delay_minutes'] > 30:
        score -= 30
    elif row['delay_minutes'] > 15:
        score -= 15
    
    # Day matching bonus
    if row['Actual_Day_Matches_Scheduled_Day'] == 1:
        score += 5
    
    return max(0, score)

air['service_score'] = air.apply(calculate_service_score, axis=1)

service_quality = air.groupby('airline').agg({
    'service_score': 'mean',
    'delay_minutes': 'mean',
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100
}).round(2)
service_quality.columns = ['avg_service_score', 'avg_delay', 'ontime_percentage']
service_quality.sort_values('avg_service_score', ascending=False)

,avg_service_score,avg_delay,ontime_percentage
airline,,,
رایمون,99.30,11.58,0.0
ماهان,95.63,18.43,0.0
پويا,95.47,17.72,0.0
آساجت,94.30,20.07,0.0
معراج,94.17,20.08,0.0
فلای پرشیا,94.11,19.04,0.0
ساها,92.95,26.11,0.0
یزد ایر,91.63,22.49,0.0
چابهار,91.26,24.06,0.0


In [203]:
# Passenger satisfaction proxy
satisfaction_factors = air.groupby('airline').agg({
    'Actual_Day_Matches_Scheduled_Day': 'mean',
    'delay_minutes': lambda x: (x <= 15).sum() / len(x) * 100,  # Acceptable delays
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100
}).round(2)
satisfaction_factors.columns = ['schedule_reliability', 'acceptable_delay_rate', 'ontime_rate']
satisfaction_factors['satisfaction_index'] = (satisfaction_factors['schedule_reliability'] * 0.3 + 
                                            satisfaction_factors['acceptable_delay_rate'] * 0.4 + 
                                            satisfaction_factors['ontime_rate'] * 0.3)
satisfaction_factors.sort_values('satisfaction_index', ascending=False)

,schedule_reliability,acceptable_delay_rate,ontime_rate,satisfaction_index
airline,,,,
رایمون,1.0,76.74,0.0,30.996
آساجت,1.0,53.49,0.0,21.696
ماهان,1.0,52.04,0.0,21.116
معراج,1.0,50.00,0.0,20.300
فلای پرشیا,1.0,48.89,0.0,19.856
پويا,1.0,46.51,0.0,18.904
قدر ایر,1.0,44.44,0.0,18.076
ساها,1.0,43.18,0.0,17.572
چابهار,1.0,42.53,0.0,17.312


## Operational Recommendations Engine

In [204]:
# Generate actionable recommendations
def generate_recommendations(airline_data):
    recommendations = []
    
    # High delay routes
    high_delay_routes = airline_data.groupby('destination_or_origin')['delay_minutes'].mean()
    if high_delay_routes.max() > 45:
        worst_route = high_delay_routes.idxmax()
        recommendations.append(f"Review operations for route to {worst_route} (avg delay: {high_delay_routes.max():.1f} min)")
    
    # Peak hour performance
    peak_hours = airline_data.groupby('Scheduled_Hour_of_Day')['delay_minutes'].mean()
    if peak_hours.max() > 30:
        worst_hour = peak_hours.idxmax()
        recommendations.append(f"Optimize scheduling for {worst_hour}:00 hour (avg delay: {peak_hours.max():.1f} min)")
    
    # Aircraft performance
    aircraft_perf = airline_data.groupby('aircraft')['delay_minutes'].mean()
    if aircraft_perf.max() > 35:
        worst_aircraft = aircraft_perf.idxmax()
        recommendations.append(f"Review {worst_aircraft} aircraft maintenance/scheduling")
    
    return recommendations

# Apply recommendations for each airline
for airline in air['airline'].unique():
    airline_data = air[air['airline'] == airline]
    recommendations = generate_recommendations(airline_data)
    print(f"\n{airline} Recommendations:")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}")


کاسپین Recommendations:
1. Review operations for route to لارستان (avg delay: 64.2 min)
2. Optimize scheduling for 17:00 hour (avg delay: 68.1 min)
3. Review 737-400 aircraft maintenance/scheduling

اطلس ایر Recommendations:
1. Review operations for route to قشم (avg delay: 176.0 min)
2. Optimize scheduling for 14:00 hour (avg delay: 186.7 min)
3. Review md83 aircraft maintenance/scheduling

آتا Recommendations:
1. Review operations for route to آبادان (avg delay: 79.2 min)
2. Optimize scheduling for 20:00 hour (avg delay: 101.6 min)
3. Review 737 aircraft maintenance/scheduling

پويا Recommendations:
1. Optimize scheduling for 16:00 hour (avg delay: 43.0 min)

ایران ایر Recommendations:
1. Review operations for route to لارستان (avg delay: 138.0 min)
2. Optimize scheduling for 6:00 hour (avg delay: 60.1 min)
3. Review AB3 aircraft maintenance/scheduling

ماهان Recommendations:
1. Review operations for route to زنجان (avg delay: 46.0 min)
2. Optimize scheduling for 4:00 hour (avg dela

## Performance Dashboard Summary

In [205]:
# Executive summary dashboard
dashboard_summary = air.groupby('airline').agg({
    'delay_minutes': ['mean', 'std'],
    'Early_OnTime_Late_Indicator': lambda x: (x == 'OnTime').sum() / len(x) * 100,
    'flight_number': 'count',
    'destination_or_origin': 'nunique',
    'estimated_cost': 'sum'
}).round(2)

dashboard_summary.columns = ['avg_delay', 'delay_consistency', 'ontime_percentage', 'total_flights', 'routes_served', 'total_cost']
dashboard_summary['performance_grade'] = pd.cut(dashboard_summary['ontime_percentage'], 
                                               bins=[0, 60, 70, 80, 90, 100], 
                                               labels=['F', 'D', 'C', 'B', 'A'])
dashboard_summary.sort_values('ontime_percentage', ascending=False)

,avg_delay,delay_consistency,ontime_percentage,total_flights,routes_served,total_cost,performance_grade
airline,,,,,,,
آتا,52.98,55.79,0.0,323,17,855700.0,NaN
آساجت,20.07,23.17,0.0,43,8,43150.0,NaN
آسمان,53.69,82.90,0.0,61,13,163750.0,NaN
آوا,41.58,47.69,0.0,24,4,49900.0,NaN
اروان,26.18,15.29,0.0,11,1,14400.0,NaN
اطلس ایر,52.30,73.56,0.0,71,10,185650.0,NaN
ایران ایر,35.03,53.12,0.0,122,30,213700.0,NaN
ایران ایرتور,44.63,60.24,0.0,98,12,218700.0,NaN
تابان,42.08,38.47,0.0,25,3,52600.0,NaN


In [206]:
# Key Performance Indicators (KPIs)
kpi_summary = {
    'Total Flights Analyzed': len(air),
    'Overall On-Time Performance': f"{(air['Early_OnTime_Late_Indicator'] == 'OnTime').sum() / len(air) * 100:.1f}%",
    'Average Delay': f"{air['delay_minutes'].mean():.1f} minutes",
    'Total Estimated Cost': f"${air['estimated_cost'].sum():,.0f}",
    'Flights Requiring Compensation': air['compensation_required'].sum(),
    'Most Problematic Hour': f"{air.groupby('Scheduled_Hour_of_Day')['delay_minutes'].mean().idxmax()}:00",
    'Best Performing Airline': dashboard_summary.index[0],
    'Worst Performing Route': air.groupby('destination_or_origin')['delay_minutes'].mean().idxmax()
}

print("=== FLIGHT OPERATIONS KPI DASHBOARD ===")
for kpi, value in kpi_summary.items():
    print(f"{kpi}: {value}")

=== FLIGHT OPERATIONS KPI DASHBOARD ===
Total Flights Analyzed: 2116
Overall On-Time Performance: 0.0%
Average Delay: 35.4 minutes
Total Estimated Cost: $3,740,100
Flights Requiring Compensation: 106
Most Problematic Hour: 1:00
Best Performing Airline: آتا
Worst Performing Route: لارستان


In [215]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = [
            "_".join(str(x) for x in col if x not in (None, "")) 
            for col in df.columns.values
        ]
    return df

def _convert_frozensets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.select_dtypes(include="object"):
        df[col] = df[col].apply(
            lambda x: ",".join(sorted(x)) if isinstance(x, (set, frozenset)) else x
        )
    return df

def save_all_dfs(
    path: str = "all_tables.xlsx",
    index: bool = False,
    dfs: dict = None
):
    """
    Save each DataFrame in `dfs` (or in globals() if None) to its own sheet.
    Skips any DataFrame that's completely empty, and reports errors per sheet.
    """
    # 1) Collect
    if dfs is None:
        dfs = {
            name: obj
            for name, obj in globals().items()
            if isinstance(obj, pd.DataFrame)
        }
    # 2) Drop empty ones
    dfs = {
        name: df
        for name, df in dfs.items()
        if df.shape[0] and df.shape[1]
    }
    if not dfs:
        print("No non-empty DataFrames found. Exiting.")
        return

    # 3) Create workbook and remove default sheet
    wb = Workbook()
    for s in wb.sheetnames:
        wb.remove(wb[s])

    used = set()
    for name, df in dfs.items():
        # 4) sanitize sheet name (<=31 chars, unique, non-empty)
        base = "".join(c for c in name if c.isalnum()) or "Sheet"
        sheet = base[:28]
        i = 1
        while sheet in used:
            sheet = f"{base[:25]}_{i}"
            i += 1
        used.add(sheet)

        # 5) prep and write, catching any errors
        try:
            df2 = _flatten_columns(df)
            df2 = _convert_frozensets(df2)

            ws = wb.create_sheet(title=sheet)
            for row in dataframe_to_rows(df2, index=index, header=True):
                ws.append(row)

        except Exception as e:
            print(f"⚠️ Skipped DataFrame '{name}' (sheet '{sheet}'): {e}")

    # 6) save
    wb.save(path)
    print(f"✅ Saved {len(used)} sheets to '{path}'")

# ---- USAGE ----
# Put this at the very end of your notebook, then just run:
save_all_dfs("flight_delay_analysis_tables.xlsx", index=False)


Exception ignored in: <function ZipFile.__del__ at 0x78cfb277fec0>
Traceback (most recent call last):
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1940, in __del__
    self.close()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1957, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


✅ Saved 189 sheets to 'flight_delay_analysis_tables.xlsx'


In [220]:
import os
import csv
import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

def _sanitize_name(name: str) -> str:
    # keep only alphanumerics/underscore, truncate to 50 chars
    safe = "".join(c if c.isalnum() or c=='_' else '_' for c in name)
    return (safe[:50] or "table")

def _prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    # 1) Flatten MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = [
            "_".join(str(x) for x in col if x not in (None, "")) 
            for col in df.columns.values
        ]
    # 2) stringify sets/frozensets
    for c in df.select_dtypes(include="object"):
        df[c] = df[c].apply(
            lambda x: ",".join(sorted(x)) if isinstance(x, (set, frozenset)) else x
        )
    # 3) force columns → plain Python strings
    df.columns = list(map(str, df.columns))
    # 4) reset to RangeIndex
    df = df.reset_index(drop=True)
    return df

def save_each_df(
    csv_dir: str = "../data/EDA/csv",
    xlsx_dir: str = "../data/EDA/xlsx"
):
    os.makedirs(csv_dir, exist_ok=True)
    os.makedirs(xlsx_dir, exist_ok=True)

    # collect only DataFrames whose names start with a letter (skip _private ones)
    dfs = {
        name: obj for name, obj in globals().items()
        if (
            isinstance(obj, pd.DataFrame)
            and name and name[0].isalpha()
            and obj.shape[0] and obj.shape[1]
        )
    }
    if not dfs:
        print("▶️ No DataFrames found to save.")
        return

    for name, df in dfs.items():
        safe = _sanitize_name(name)
        try:
            df2 = _prepare_df(df)

            # --- CSV (manual) ---
            csv_path = os.path.join(csv_dir, f"{safe}.csv")
            with open(csv_path, "w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(df2.columns)         # header
                for row in df2.itertuples(index=False, name=None):
                    writer.writerow(row)

            # --- XLSX ---
            xlsx_path = os.path.join(xlsx_dir, f"{safe}.xlsx")
            wb = Workbook()
            wb.remove(wb.active)
            ws = wb.create_sheet(title=safe[:31])
            for r in dataframe_to_rows(df2, index=False, header=True):
                ws.append(r)
            wb.save(xlsx_path)

            print(f"✅ Saved `{name}`:")
            print(f"     • CSV → {csv_path}")
            print(f"     • XLSX → {xlsx_path}")

        except Exception as e:
            print(f"⚠️ Skipped `{name}`: {e}")

# Usage: put this cell at the end, then run:
save_each_df()


✅ Saved `df`:
     • CSV → ../data/EDA/csv/df.csv
     • XLSX → ../data/EDA/xlsx/df.xlsx
✅ Saved `air`:
     • CSV → ../data/EDA/csv/air.csv
     • XLSX → ../data/EDA/xlsx/air.xlsx
✅ Saved `df_assoc`:
     • CSV → ../data/EDA/csv/df_assoc.csv
     • XLSX → ../data/EDA/xlsx/df_assoc.xlsx
✅ Saved `df_onehot`:
     • CSV → ../data/EDA/csv/df_onehot.csv
     • XLSX → ../data/EDA/xlsx/df_onehot.xlsx
✅ Saved `frequent_itemsets`:
     • CSV → ../data/EDA/csv/frequent_itemsets.csv
     • XLSX → ../data/EDA/xlsx/frequent_itemsets.xlsx
✅ Saved `rules`:
     • CSV → ../data/EDA/csv/rules.csv
     • XLSX → ../data/EDA/xlsx/rules.xlsx
✅ Saved `late_rules`:
     • CSV → ../data/EDA/csv/late_rules.csv
     • XLSX → ../data/EDA/xlsx/late_rules.xlsx
✅ Saved `late_rules_freeze`:
     • CSV → ../data/EDA/csv/late_rules_freeze.csv
     • XLSX → ../data/EDA/xlsx/late_rules_freeze.xlsx
✅ Saved `late_rules_by_day`:
     • CSV → ../data/EDA/csv/late_rules_by_day.csv
     • XLSX → ../data/EDA/xlsx/late_rules_b